# Semi-Supervised CIFAR-10 Classification — CLIP-based Pipeline (v4.1, 高速化版)

**v4.1 の変更点 (v4 → v4.1): 実行時間の異常な長さ (Stage1b: 6h, Stage3: 8h) を修正**

判明した原因:
1. **Stage 1b**: zero-shot 80プロンプトのテキスト埋め込みを pseudo-label refresh ごと
   (200epoch中10回) に毎回再計算していた → クラス名は不変なので1度だけ計算してキャッシュするよう修正
2. **Stage 3**: 1バッチにつき CLIP ViT-L/14 の forward を4回呼んでおり、
   200epoch × 約53batch × 4回 ≒ 42,400回 のCLIP forwardが発生していた
   → Stage 1完了後はCLIPが完全frozenであることを利用し、学習開始前に
     identity + weak×2 + strong×4 のCLIP特徴を1度だけ計算してキャッシュ。
     Stage 1b・Stage 3・推論(TTA)のいずれもCLIP forwardを呼ばず、
     キャッシュからのlookup + 軽量Adapter/MLPのみで完結するように変更。

**期待効果:** Stage 1b・Stage 3 ともに数分〜数十分程度まで短縮 (CLIP forward回数が
ほぼゼロになるため、ボトルネックがMLPの学習のみになる)。

**Flow (v4と同じ構成、計算経路のみ変更):**
- Stage 1: CLIP ViT-L/14 (最終2ブロックLLRD fine-tune可) + Adapter 対照学習
- **[NEW] Stage 1完了後: 特徴キャッシュ構築** (identity/weak×2/strong×4、CLIPは以降使わない)
- Stage 1b: 疑似ラベリング (GCN + CLIP zero-shot 80プロンプト、すべてキャッシュ経由)
- Stage 2: CLIP特徴空間 Manifold Mixup (キャッシュ済みadapt後特徴を使用)
- Stage 3: 分類器ヘッド学習 (キャッシュからのlookupのみ、CLIP forwardなし)
- Inference: TTA (test_cacheの全variantを使用) + SWA/EMA
- 比較: pretrained ResNet-50 fine-tune (画像を直接使用) / CLIP supervised-only (キャッシュ使用)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset
from torch.optim.swa_utils import AveragedModel, SWALR
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import numpy as np
import random
import math
import copy
import os
from dataclasses import dataclass, field
from typing import Optional, Tuple, List
from tqdm.auto import tqdm
import ssl
import warnings
warnings.filterwarnings("ignore")
ssl._create_default_https_context = ssl._create_unverified_context

# CLIP (open_clip): pip install open_clip_torch
import open_clip

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


## Config

In [ ]:
@dataclass
class Config:
    # ============================================================
    # Data
    # ============================================================
    n_labeled:        int   = 500
    n_unlabeled:      int   = 3000
    n_classes:        int   = 10
    img_size:         int   = 224    # CLIP ViT-L/14 の入力解像度
    val_ratio:        float = 0.2

    cifar10_classes: Tuple[str, ...] = (
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck")

    # ============================================================
    # Stage 1: CLIP backbone + LLRD fine-tune + Adapter
    # ============================================================
    clip_model_name:  str   = "ViT-L-14"    # v3: ViT-B-32 -> v4: ViT-L-14 (+2~4%)
    clip_pretrained:  str   = "openai"
    clip_feat_dim:    int   = 768            # ViT-L/14 の出力次元

    # LLRD (Layer-wise LR Decay): CLIPの最終Nブロックのみ低LRでfine-tune
    # 0 = 完全frozen (v3相当), 2 = 最終2ブロック解凍 (推奨)
    clip_finetune_blocks: int   = 2
    clip_finetune_lr:     float = 1e-5      # CLIPのfine-tune用LR (head LRの1/100程度)
    clip_finetune_wd:     float = 1e-4

    # 特徴キャッシュ (Stage 1完了後、CLIPは完全frozenになるため特徴を事前計算する)
    n_weak_variants:  int   = 2     # weak aug のバリエーション数
    n_strong_variants: int  = 4     # strong aug のバリエーション数
    cache_batch:      int   = 64    # キャッシュ作成時のCLIP forwardバッチサイズ

    # Adapter
    adapter_hidden:   int   = 256
    adapter_out:      int   = 128
    adapter_epochs:   int   = 60
    adapter_lr:       float = 1e-3
    adapter_batch:    int   = 64            # ViT-L/14はVRAM消費大のためbatchを縮小
    adapter_temp:     float = 0.5

    # ============================================================
    # Stage 1b: Pseudo-labeling
    # GCN + CLIP zero-shot 80プロンプトアンサンブルに固定
    # ============================================================
    lp_k:             int   = 15
    lp_alpha:         float = 0.90
    lp_iters:         int   = 60
    lp_topk_init:     int   = 20
    lp_topk_max:      int   = 60
    lp_conf_floor:    float = 0.55

    gcn_hidden:       int   = 128
    gcn_epochs:       int   = 150
    gcn_lr:           float = 5e-3
    gcn_dropout:      float = 0.3

    zs_ensemble_w:    float = 0.4    # GCN:(1-w)  zero-shot:w

    pseudo_refresh:   int   = 20

    # ============================================================
    # Stage 2: CLIP特徴空間 Manifold Mixup
    # ============================================================
    use_feature_mixup: bool  = True
    mixup_alpha:       float = 0.4
    mixup_pairs_per_epoch: int = 4

    # ============================================================
    # Stage 3: 分類器ヘッド
    # ============================================================
    cls_hidden:       int   = 512           # ViT-L/14の768次元に合わせて拡大
    cls_dropout:      float = 0.2
    cls_epochs:       int   = 200
    cls_lr:           float = 1e-3
    cls_batch:        int   = 64
    fixmatch_thresh:  float = 0.85
    fixmatch_temp:    float = 0.5
    label_smoothing:  float = 0.1

    lam_pseudo:       float = 0.5
    lam_mixup:        float = 0.5

    ema_decay:        float = 0.999
    swa_start:        float = 0.75
    log_interval:     int   = 10

    device: str = DEVICE

cfg = Config()


## データ読み込み (CIFAR-10, stratified labeled/unlabeled/val split)

In [ ]:
class CIFARSemiDataset(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor):
        self.images = images
        self.labels = labels
    def __len__(self): return len(self.images)
    def __getitem__(self, i): return self.images[i], self.labels[i]


def load_cifar10(cfg: Config):
    """
    CIFAR-10 を読み込み、labeled を stratified に train_l / val_l へ分割。
    unlabeled は labeled とは独立したサブセットから sampling。
    test は返すが最後の評価にのみ使う。

    画像は CLIP の前処理 (Resize 224 + CLIP正規化) に合わせる。
    augmentation (weak/strong) は別セルで定義し、ここでは「素の」テンソルのみ保持する
    (0-1 range, リサイズ後)。CLIP正規化は特徴抽出関数の内部で行う。
    """
    tf = T.Compose([
        T.Resize(cfg.img_size),
        T.ToTensor(),
    ])
    full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
    test_ds    = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf)

    per_class = cfg.n_labeled // cfg.n_classes
    labeled_idx, unlabeled_idx = [], []
    counts = {c: 0 for c in range(cfg.n_classes)}
    perm = torch.randperm(len(full_train)).tolist()
    for i in perm:
        _, lbl = full_train[i]
        if counts[lbl] < per_class:
            labeled_idx.append(i); counts[lbl] += 1
        elif len(unlabeled_idx) < cfg.n_unlabeled:
            unlabeled_idx.append(i)
        if len(labeled_idx) == cfg.n_labeled and len(unlabeled_idx) == cfg.n_unlabeled:
            break

    def collect(ds, idx, mask_label=False):
        imgs, lbls = [], []
        for i in idx:
            im, lb = ds[i]
            imgs.append(im)
            lbls.append(-1 if mask_label else lb)
        return torch.stack(imgs), torch.tensor(lbls)

    lx, ly = collect(full_train, labeled_idx)
    ux, _  = collect(full_train, unlabeled_idx, mask_label=True)

    # --- Stratified train/val split of labeled data ---
    val_per_class = max(1, int(per_class * cfg.val_ratio))
    train_idx, val_idx = [], []
    vcounts = {c: 0 for c in range(cfg.n_classes)}
    lperm = torch.randperm(len(lx)).tolist()
    for i in lperm:
        c = ly[i].item()
        if vcounts[c] < val_per_class:
            val_idx.append(i); vcounts[c] += 1
        else:
            train_idx.append(i)

    lx_train, ly_train = lx[train_idx], ly[train_idx]
    lx_val,   ly_val   = lx[val_idx],   ly[val_idx]

    # test (10000件) はメモリ節約のため必要時にバッチで処理する想定だが、
    # ここでは MNIST版と同様に一括 tensor 化する (CIFAR-10 32x32->224x224 でも
    # 10000枚 * 3*224*224*4bytes ≈ 6GB弱になるため、float16 で保持しメモリを抑える)。
    test_imgs = torch.stack([test_ds[i][0] for i in range(len(test_ds))]).half()
    test_lbls = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

    print(f"Labeled train: {len(lx_train)}, Labeled val: {len(lx_val)}, "
          f"Unlabeled: {len(ux)}, Test: {len(test_imgs)}")
    return lx_train, ly_train, lx_val, ly_val, ux, test_imgs, test_lbls

lx_train, ly_train, lx_val, ly_val, ux, test_imgs, test_lbls = load_cifar10(cfg)


## Augmentation (Weak / Strong, FixMatch式分離)

In [ ]:
# ============================================================
# Augmentation
# ------------------------------------------------------------
# v2 (MNIST/GAN版) では DiffAugment + ADA + weak/strong 分離を
# Discriminator の学習安定化のために使っていたが、v3 では GAN を
# 廃止したため、augmentation の役割は以下の2つに単純化される:
#   weak   : 疑似ラベルの「決定」に使う (FixMatchの思想を継続)
#   strong : 実際の分類器学習に使う (RandAugment相当)
# CLIP の事前学習分布を大きく破壊しないよう、color jitter は
# 控えめにする (CLIPは色情報への依存が大きいため過度な変形は逆効果)。
# ============================================================

_weak_tf = T.Compose([
    T.RandomCrop(cfg.img_size, padding=16, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
])

_strong_tf = T.Compose([
    T.RandomCrop(cfg.img_size, padding=16, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.3, 0.3, 0.3, 0.05)], p=0.8),
    T.RandomGrayscale(p=0.1),
])


def weak_aug(x: torch.Tensor) -> torch.Tensor:
    x = x.float()
    return torch.stack([_weak_tf(img) for img in x])


def strong_aug(x: torch.Tensor) -> torch.Tensor:
    x = x.float()
    return torch.stack([_strong_tf(img) for img in x])


# ---------- Mixup / CutMix (画像レベル; 分類器学習の補助として残す) ----------
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    cr = math.sqrt(1 - lam); ch = int(H * cr); cw = int(W * cr)
    cx = random.randint(0, W); cy = random.randint(0, H)
    x1 = max(0, cx - cw // 2); x2 = min(W, cx + cw // 2)
    y1 = max(0, cy - ch // 2); y2 = min(H, cy + ch // 2)
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    idx = torch.randperm(B, device=x.device)
    out = x.clone(); out[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    return out, y, y[idx], lam

def mix_criterion(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1 - lam) * crit(pred, yb)


def get_cls_view(x, y, epoch, cfg, total_epochs):
    """Curriculum: 序盤は素の weak aug、後半に mixup/cutmix を混ぜる。"""
    x = weak_aug(x)
    frac = epoch / max(1, total_epochs)
    if frac < 0.3:
        return x, y, y, 1.0
    if random.random() < 0.5:
        fn = mixup if random.random() < 0.5 else cutmix
        return fn(x, y)
    return x, y, y, 1.0


## Stage 1: CLIP backbone (frozen) + 軽量 Adapter の対照学習

In [ ]:
# ================================================================
# CLIP backbone (ViT-L/14) + LLRD fine-tune + 軽量 Adapter
# ----------------------------------------------------------------
# v4 変更点:
#   1. ViT-B/32 -> ViT-L/14: 特徴次元512->768, patch 32->14, 表現力大幅向上
#   2. LLRD (Layer-wise LR Decay): CLIPを完全凍結せず最終Nブロックだけ
#      低LR (1e-5) でfine-tune。500枚でも上位層のみなら過学習しにくい。
#   3. Adapterはv3と同様 (CLIP特徴の上のresidual MLP)
# ================================================================

class CLIPBackbone(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms(
            cfg.clip_model_name, pretrained=cfg.clip_pretrained)
        self.visual = model.visual
        self.feat_dim = cfg.clip_feat_dim

        # まず全部凍結
        for p in self.visual.parameters():
            p.requires_grad_(False)

        # LLRD: 最終 clip_finetune_blocks 個の transformer ブロックのみ解凍
        # ViT-L/14 では visual.transformer.resblocks が transformer層リスト
        if cfg.clip_finetune_blocks > 0:
            try:
                blocks = list(self.visual.transformer.resblocks)
                for blk in blocks[-cfg.clip_finetune_blocks:]:
                    for p in blk.parameters():
                        p.requires_grad_(True)
                # 最終層ノルム・投影も解凍
                for attr in ("ln_post", "proj"):
                    m = getattr(self.visual, attr, None)
                    if m is not None:
                        if hasattr(m, "parameters"):
                            for p in m.parameters(): p.requires_grad_(True)
                        elif isinstance(m, torch.Tensor):
                            m.requires_grad_(True)
                n_trainable = sum(p.numel() for p in self.visual.parameters() if p.requires_grad)
                print(f"[CLIP LLRD] fine-tune last {cfg.clip_finetune_blocks} blocks "
                      f"({n_trainable/1e6:.1f}M params)")
            except AttributeError:
                print("[CLIP LLRD] resblocks not found, keeping fully frozen")

        self.register_buffer(
            "mean", torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1,3,1,1))
        self.register_buffer(
            "std",  torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1,3,1,1))

    def forward(self, x):
        """x: (B,3,H,W) in [0,1], any dtype. 正規化してCLIP特徴を返す。"""
        x = x.float()
        x = (x - self.mean) / self.std
        return self.visual(x).float()

    def frozen_forward(self, x):
        """frozen部分のみの forward (Adapterの対照学習に使用)。"""
        with torch.no_grad():
            return self.forward(x)

    def get_finetune_params(self, lr, wd):
        """LLRD対象パラメータのみを optimizer に渡すための param group。"""
        return [{"params": [p for p in self.visual.parameters() if p.requires_grad],
                 "lr": lr, "weight_decay": wd}]


class Adapter(nn.Module):
    """CLIP特徴 -> residual MLP -> 対照学習用projection (v3と同じ設計)。"""
    def __init__(self, cfg):
        super().__init__()
        d = cfg.clip_feat_dim
        self.net = nn.Sequential(
            nn.Linear(d, cfg.adapter_hidden), nn.ReLU(True),
            nn.Linear(cfg.adapter_hidden, d))
        self.projector = nn.Sequential(
            nn.Linear(d, d), nn.ReLU(True),
            nn.Linear(d, cfg.adapter_out))
        self.residual_scale = nn.Parameter(torch.tensor(0.1))

    def adapt(self, clip_feat):
        return clip_feat + self.residual_scale * self.net(clip_feat)

    def forward(self, clip_feat):
        return F.normalize(self.projector(self.adapt(clip_feat)), dim=-1)


class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.T = temperature
    def forward(self, z1, z2):
        N = z1.size(0); z = torch.cat([z1, z2], 0)
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=-1) / self.T
        sim.masked_fill_(torch.eye(2*N, device=z.device).bool(), -1e9)
        tgt = torch.cat([torch.arange(N, device=z.device) + N,
                         torch.arange(N, device=z.device)])
        return F.cross_entropy(sim, tgt)


@torch.no_grad()
def extract_clip_features(clip_model, imgs, batch=64, device=DEVICE):
    """画像 -> CLIP特徴 (CPU float32)。ViT-L/14はVRAM大きいのでbatch=64推奨。"""
    clip_model.eval()
    feats = []
    for i in range(0, len(imgs), batch):
        x = imgs[i:i+batch].float().to(device)
        feats.append(clip_model(x).cpu())
    return torch.cat(feats, 0)


def train_adapter(cfg, clip_model, lx_train, ux):
    """
    Stage 1: Adapter + CLIP最終ブロックの同時対照学習。
    Adapterは全パラメータ更新、CLIPはLLRDで解凍されたブロックのみ低LRで更新。
    """
    all_imgs = torch.cat([lx_train, ux], 0)
    adapter = Adapter(cfg).to(cfg.device)
    crit    = NTXentLoss(cfg.adapter_temp)

    os.makedirs("models/sgan", exist_ok=True)
    save_adapter = "models/sgan/adapter_best.pth"
    save_clip    = "models/sgan/clip_finetune_best.pth"
    if os.path.exists(save_adapter):
        tqdm.write(f"Loading Adapter from {save_adapter}")
        adapter.load_state_dict(torch.load(save_adapter, map_location=cfg.device, weights_only=True))
        if os.path.exists(save_clip) and cfg.clip_finetune_blocks > 0:
            clip_model.visual.load_state_dict(
                torch.load(save_clip, map_location=cfg.device, weights_only=True))
            tqdm.write(f"Loading CLIP fine-tune from {save_clip}")
        return adapter

    # Optimizer: Adapterは通常LR、CLIPのfine-tune部分は低LR
    param_groups = [{"params": adapter.parameters(), "lr": cfg.adapter_lr, "weight_decay": 1e-6}]
    if cfg.clip_finetune_blocks > 0:
        param_groups += clip_model.get_finetune_params(cfg.clip_finetune_lr, cfg.clip_finetune_wd)
    opt   = torch.optim.AdamW(param_groups)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.adapter_epochs)

    loader = DataLoader(TensorDataset(all_imgs), batch_size=cfg.adapter_batch,
                        shuffle=True, num_workers=0, pin_memory=True, drop_last=True)

    best_loss = float("inf")
    epoch_bar = tqdm(range(cfg.adapter_epochs), desc="Adapter", position=0, leave=True, unit="ep")
    for epoch in epoch_bar:
        adapter.train()
        if cfg.clip_finetune_blocks > 0:
            clip_model.train()   # fine-tuneブロックを train mode にする
        total = 0.0
        for (x,) in loader:
            x1 = weak_aug(x).to(cfg.device)
            x2 = strong_aug(x).to(cfg.device)
            # LLRD対象は勾配あり、frozenは no_grad
            f1 = clip_model(x1)
            f2 = clip_model(x2)
            z1 = adapter(f1); z2 = adapter(f2)
            loss = crit(z1, z2)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
            epoch_bar.set_postfix(loss=f"{loss.item():.4f}")
        sched.step()
        avg = total / len(loader)
        if (epoch+1) % 10 == 0:
            tqdm.write(f"[Adapter ep {epoch+1:03d}/{cfg.adapter_epochs}] loss={avg:.4f}")
        if avg < best_loss:
            best_loss = avg
            torch.save(adapter.state_dict(), save_adapter)
            if cfg.clip_finetune_blocks > 0:
                torch.save(clip_model.visual.state_dict(), save_clip)

    adapter.load_state_dict(torch.load(save_adapter, map_location=cfg.device, weights_only=True))
    if os.path.exists(save_clip) and cfg.clip_finetune_blocks > 0:
        clip_model.visual.load_state_dict(
            torch.load(save_clip, map_location=cfg.device, weights_only=True))
    clip_model.eval()
    for p in clip_model.visual.parameters():
        p.requires_grad_(False)   # 学習後は再度凍結 (分類器学習時はfrozenで使う)
    return adapter


print("=== Stage 1: CLIP ViT-L/14 + LLRD fine-tune + Adapter ===")
clip_model = CLIPBackbone(cfg).to(cfg.device)
adapter    = train_adapter(cfg, clip_model, lx_train, ux)

# ================================================================
# Feature Cache (v4.1: Stage 1完了後にCLIPを使い切ってキャッシュする)
# ----------------------------------------------------------------
# Stage 1終了時点でCLIP (LLRD込み) は完全 frozen に固定される。
# 以降、CLIPの出力は「入力画像 (とそのaugmentation) が決まれば不変」になるため、
# Stage 1b / Stage 3 で同じ画像に何度もCLIP forwardを通すのは無駄である。
#
# 解決策: 学習開始前に、画像ごとに
#   - identity (無augmentation) 特徴 ×1
#   - weak aug 特徴            ×n_weak_variants
#   - strong aug 特徴          ×n_strong_variants
# をあらかじめ計算し、Adapter適用済みの特徴として保持する。
# Stage 3の学習ループでは、この中からランダムに1つを選んで使うことで
# CLIP forward を完全に省略できる (Adapterも軽量MLPなのでCPU/GPUどちらでも高速)。
#
# 注意: Adapter自体は Stage 1 で学習済みの重みを使うが、Adapterは小さいMLPなので
# 「CLIP特徴(adapt前)」をキャッシュし、adapter.adapt()は都度(軽量に)適用する方式にする。
# これにより、Adapterを将来再学習したくなった場合でもキャッシュを再利用できる。
# ================================================================

class FeatureCache:
    """
    画像セット (labeled+unlabeled, またはtest) に対する
    {identity, weak×N, strong×M} の CLIP特徴 (adapt前、768次元) をまとめて保持する。
    """
    def __init__(self, identity_feat, weak_feats, strong_feats):
        self.identity = identity_feat         # (N, D)
        self.weak     = weak_feats            # (n_weak, N, D)
        self.strong   = strong_feats          # (n_strong, N, D)
        self.N = identity_feat.size(0)

    def sample_weak(self, idx):
        v = random.randrange(self.weak.size(0))
        return self.weak[v, idx]

    def sample_strong(self, idx):
        v = random.randrange(self.strong.size(0))
        return self.strong[v, idx]

    def get_identity(self, idx):
        return self.identity[idx]


@torch.no_grad()
def build_feature_cache(cfg, clip_backbone, imgs, desc="cache"):
    """
    imgs: (N,3,H,W) の画像テンソル。CLIPは完全frozenであることを前提とする。
    identity + weak×n_weak_variants + strong×n_strong_variants のCLIP特徴 (adapt前) を計算する。
    """
    clip_backbone.eval()
    N = len(imgs)

    def extract(transform_fn):
        feats = []
        for i in range(0, N, cfg.cache_batch):
            x = imgs[i:i+cfg.cache_batch]
            x = transform_fn(x) if transform_fn is not None else x.float()
            feats.append(clip_backbone(x.to(cfg.device)).cpu())
        return torch.cat(feats, 0)

    identity_feat = extract(None)

    weak_list = []
    for v in tqdm(range(cfg.n_weak_variants), desc=f"{desc} (weak)", leave=False):
        weak_list.append(extract(weak_aug))
    weak_feats = torch.stack(weak_list, 0)

    strong_list = []
    for v in tqdm(range(cfg.n_strong_variants), desc=f"{desc} (strong)", leave=False):
        strong_list.append(extract(strong_aug))
    strong_feats = torch.stack(strong_list, 0)

    return FeatureCache(identity_feat, weak_feats, strong_feats)


## Stage 1完了後: 特徴キャッシュの構築 (高速化の核心)

CLIPは以降完全frozenであることが確定したため、labeled/unlabeled/test の全画像について
`identity + weak×N + strong×M` のCLIP特徴 (adapt前) を一度だけ計算してメモリに保持する。
これ以降、Stage 1b・Stage 3・推論のいずれもCLIP forwardを呼ばず、このキャッシュから
ランダムに特徴を取り出して `adapter.adapt()` (軽量MLP) を通すだけで済む。

In [ ]:
print("=== Building feature cache (labeled+unlabeled, test) ===")
all_train_imgs = torch.cat([lx_train, ux], 0)
N_l_global = len(lx_train)

train_cache = build_feature_cache(cfg, clip_model, all_train_imgs, desc="train+unlabeled")
val_cache   = build_feature_cache(cfg, clip_model, lx_val,   desc="val")
test_cache  = build_feature_cache(cfg, clip_model, test_imgs, desc="test")

print(f"  train_cache : identity{tuple(train_cache.identity.shape)}  "
      f"weak{tuple(train_cache.weak.shape)}  strong{tuple(train_cache.strong.shape)}")
print(f"  val_cache   : identity{tuple(val_cache.identity.shape)}")
print(f"  test_cache  : identity{tuple(test_cache.identity.shape)}")


## Stage 1b: 疑似ラベリング (学習可能GCN + CLIP zero-shot アンサンブル)

In [ ]:
# ================================================================
# Stage 1b: 疑似ラベリング (学習可能GCN + CLIP zero-shot 80プロンプトアンサンブル)
# ----------------------------------------------------------------
# v4.1 変更点 (高速化):
#   1. zero-shot テキスト埋め込みはクラス名が不変なので、初回に1度だけ計算して
#      キャッシュし、以降の refresh では再計算しない (テキストエンコードのコストを除去)
#   2. 画像側もキャッシュ済み identity 特徴 (train_cache.identity) をそのまま使う
#      (CLIP forwardを毎回呼ばない)
# ================================================================

from scipy.sparse import csr_matrix

IMAGENET_TEMPLATES = [
    "a photo of a {}.", "a blurry photo of a {}.", "a black and white photo of a {}.",
    "a low contrast photo of a {}.", "a high contrast photo of a {}.", "a bad photo of a {}.",
    "a good photo of a {}.", "a photo of a small {}.", "a photo of a big {}.",
    "a photo of the {}.", "a dark photo of the {}.", "a photo of my {}.",
    "i love my {}!", "a photo of my dirty {}.", "a photo of my clean {}.",
    "a photo of my cool {}.", "a close-up photo of a {}.", "a bright photo of a {}.",
    "a cropped photo of a {}.", "a photo of the hard to see {}.",
    "a low resolution photo of the {}.", "a rendering of a {}.", "a bad photo of the {}.",
    "a cropped photo of the {}.", "a bright photo of the {}.", "a photo of the clean {}.",
    "a photo of a large {}.", "a photo of a nice {}.", "a photo of a weird {}.",
    "a photo of a cool {}.", "a close-up photo of the {}.", "a good photo of the {}.",
    "a photo of one {}.", "a photo of a small {}.", "a rendering of the {}.",
    "a photo of many {}.", "a photo of the dirty {}.", "a photo of a dirty {}.",
    "a dark photo of a {}.", "a photo of the cool {}.", "a photo of the nice {}.",
    "a photo of a large {}.", "a photo of the large {}.",
    "a black and white photo of the {}.", "a high contrast photo of the {}.",
    "a low contrast photo of the {}.", "a photo of the blurry {}.",
    "a photo of the small {}.", "a photo of the weird {}.", "an embroidered {}.",
    "a cartoon {}.", "art of the {}.", "a painting of the {}.",
    "a photo of the ugly {}.", "a photo of the cute {}.", "a photo of the large {}.",
    "a sketch of a {}.", "a drawing of a {}.", "a photo of the bad {}.",
    "an origami {}.", "a toy {}.", "a plush {}.", "a photo of a hard to see {}.",
    "a photo of a cute {}.", "a photo of a ugly {}.", "a photo of a fast {}.",
    "a photo of a slow {}.", "a photo of a tall {}.", "a photo of a short {}.",
    "a photo of a old {}.", "a photo of a new {}.", "a photo of a red {}.",
    "a photo of a blue {}.", "a photo of a green {}.", "a photo of a black {}.",
    "a photo of a white {}.", "a photo of a {}", "the {} in a video game.",
    "a photo of a {} in the wild.", "a photo of {} in the wild.",
]


# ---------- (a) Classic Label Propagation (ablation用に保持) ----------
def classic_label_propagation(cfg, feats_np, ly_train, N_l):
    N = feats_np.shape[0]
    sim = feats_np @ feats_np.T
    np.fill_diagonal(sim, -1)
    knn_idx = np.argsort(-sim, axis=1)[:, :cfg.lp_k]
    knn_val = np.exp(sim[np.arange(N)[:, None], knn_idx])
    rows = np.repeat(np.arange(N), cfg.lp_k); cols = knn_idx.ravel(); vals = knn_val.ravel()
    W = csr_matrix((vals, (rows, cols)), shape=(N,N)); W = (W + W.T) / 2
    d = np.asarray(W.sum(1)).ravel() ** -0.5
    D_mat = csr_matrix((d, (np.arange(N), np.arange(N))), shape=(N,N))
    S = (D_mat @ W @ D_mat).toarray().astype(np.float32)
    Y = np.zeros((N, cfg.n_classes), dtype=np.float32)
    for i, lbl in enumerate(ly_train.numpy()):
        Y[i, lbl] = 1.0
    F_mat = Y.copy()
    for _ in range(cfg.lp_iters):
        F_mat = cfg.lp_alpha * (S @ F_mat) + (1 - cfg.lp_alpha) * Y
    F_mat /= (F_mat.sum(1, keepdims=True) + 1e-8)
    return F_mat


# ---------- (b) 学習可能 GCN ----------
class LabelPropGCN(nn.Module):
    def __init__(self, in_dim, hidden, n_classes, dropout=0.3):
        super().__init__()
        self.gc1 = nn.Linear(in_dim, hidden)
        self.gc2 = nn.Linear(hidden, n_classes)
        self.dropout = dropout

    def forward(self, x, S):
        h = S @ x
        h = F.relu(self.gc1(h))
        h = F.dropout(h, self.dropout, training=self.training)
        h = S @ h
        return self.gc2(h)


def build_knn_graph(feats_np, k):
    N = feats_np.shape[0]
    sim = feats_np @ feats_np.T
    np.fill_diagonal(sim, -1)
    knn_idx = np.argsort(-sim, axis=1)[:, :k]
    knn_val = np.exp(sim[np.arange(N)[:, None], knn_idx])
    rows = np.repeat(np.arange(N), k); cols = knn_idx.ravel(); vals = knn_val.ravel()
    W = csr_matrix((vals, (rows, cols)), shape=(N,N)); W = (W + W.T) / 2
    d = np.asarray(W.sum(1)).ravel() ** -0.5
    D_mat = csr_matrix((d, (np.arange(N), np.arange(N))), shape=(N,N))
    return (D_mat @ W @ D_mat).toarray().astype(np.float32)


def gcn_label_propagation(cfg, feats_np, ly_train, N_l, device=DEVICE):
    N = feats_np.shape[0]
    S = torch.tensor(build_knn_graph(feats_np, cfg.lp_k), device=device)
    x = torch.tensor(feats_np, dtype=torch.float32, device=device)
    y = ly_train.to(device)
    model = LabelPropGCN(feats_np.shape[1], cfg.gcn_hidden, cfg.n_classes,
                         dropout=cfg.gcn_dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.gcn_lr, weight_decay=5e-4)
    best_state = None; best_loss = float("inf")
    for _ in range(cfg.gcn_epochs):
        model.train()
        logits = model(x, S)
        loss = F.cross_entropy(logits[:N_l], y)
        opt.zero_grad(); loss.backward(); opt.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return F.softmax(model(x, S), dim=-1).cpu().numpy()


# ---------- (c) CLIP zero-shot 80プロンプトアンサンブル ----------
# テキスト埋め込みはクラス名が不変なため、モジュールレベルで1度だけ計算してキャッシュする。
_zeroshot_text_cache = {}

@torch.no_grad()
def get_zeroshot_text_features(cfg, clip_model_full, device=DEVICE):
    """クラスごとの80プロンプト平均テキスト特徴を計算しキャッシュする (再計算は一切しない)。"""
    key = (cfg.clip_model_name, cfg.clip_pretrained)
    if key in _zeroshot_text_cache:
        return _zeroshot_text_cache[key]

    tokenizer = open_clip.get_tokenizer(cfg.clip_model_name)
    class_embeddings = []
    for cls_name in cfg.cifar10_classes:
        prompts = [t.format(cls_name) for t in IMAGENET_TEMPLATES]
        tokens  = tokenizer(prompts).to(device)
        feats   = clip_model_full.encode_text(tokens).float()
        feats   = F.normalize(feats, dim=-1)
        cls_emb = F.normalize(feats.mean(0, keepdim=True), dim=-1)
        class_embeddings.append(cls_emb)
    text_feat = torch.cat(class_embeddings, dim=0)
    _zeroshot_text_cache[key] = text_feat
    print(f"  [zero-shot] text features cached: {tuple(text_feat.shape)} "
          f"(80 prompts x {cfg.n_classes} classes, computed once)")
    return text_feat


@torch.no_grad()
def zero_shot_probs_from_image_feats(cfg, clip_model_full, img_feats_raw, device=DEVICE):
    """
    img_feats_raw: (N, feat_dim) の CLIP画像特徴 (adapt前, 正規化前)。
    キャッシュ済みのテキスト特徴と組み合わせて zero-shot 確率を計算する。
    画像側もキャッシュから取るため、CLIP forwardは一切発生しない。
    """
    text_feat = get_zeroshot_text_features(cfg, clip_model_full, device)
    img_feat = F.normalize(img_feats_raw.to(device), dim=-1)
    logits = 100.0 * img_feat @ text_feat.T
    return F.softmax(logits, dim=-1).cpu().numpy()


# ---------- メイン: アンサンブル疑似ラベリング (キャッシュ利用版) ----------
def ensemble_pseudo_label(cfg, adapter, clip_model_full, train_cache,
                          ly_train, N_l, N_total, refresh_count=0):
    """
    train_cache.identity (CLIP特徴, adapt前) をそのまま使う。
    CLIP forwardは一切発生しない (Stage1完了後はCLIPは固定なので識別特徴は不変)。
    """
    n_refreshes = max(1, cfg.cls_epochs // cfg.pseudo_refresh)
    frac = min(1.0, refresh_count / n_refreshes)
    topk = int(cfg.lp_topk_init + (cfg.lp_topk_max - cfg.lp_topk_init) * frac)
    n_unlabeled = N_total - N_l
    print(f"  [PL] refresh#{refresh_count}  topk={topk}/class  conf_floor={cfg.lp_conf_floor}")

    with torch.no_grad():
        adapted = adapter.adapt(train_cache.identity.to(cfg.device)).cpu().numpy()
    adapted_norm = adapted / (np.linalg.norm(adapted, axis=1, keepdims=True) + 1e-8)

    gcn_probs = gcn_label_propagation(cfg, adapted_norm, ly_train, N_l, device=cfg.device)
    zs_probs  = zero_shot_probs_from_image_feats(
        cfg, clip_model_full, train_cache.identity, device=cfg.device)

    w = cfg.zs_ensemble_w
    F_mat = (1 - w) * gcn_probs + w * zs_probs
    F_mat[:N_l] = 0.0
    F_mat[np.arange(N_l), ly_train.numpy()] = 1.0

    conf = F_mat.max(axis=1); hard = F_mat.argmax(axis=1)
    ul_conf = conf[N_l:]; ul_hard = hard[N_l:]
    confident_ul = np.zeros(n_unlabeled, dtype=bool)
    for c in range(cfg.n_classes):
        cls_idx = np.where(ul_hard == c)[0]
        if len(cls_idx) == 0: continue
        sorted_idx = cls_idx[np.argsort(-ul_conf[cls_idx])]
        for i in sorted_idx[:topk]:
            if ul_conf[i] >= cfg.lp_conf_floor:
                confident_ul[i] = True

    confident_mask = np.concatenate([np.ones(N_l, dtype=bool), confident_ul])
    print(f"  [PL] confident unlabeled: {confident_ul.sum()}/{n_unlabeled}  "
          f"total: {confident_mask.sum()}/{N_total}")
    return torch.tensor(F_mat, dtype=torch.float32), confident_mask, hard


# clip_model_full (テキストエンコーダ含む完全版)。画像エンコーダ部分は使わない
# (画像特徴はキャッシュから取るため、テキストエンコーダの利用のみ)。
_clip_full, _, _ = open_clip.create_model_and_transforms(
    cfg.clip_model_name, pretrained=cfg.clip_pretrained)
_clip_full = _clip_full.to(cfg.device).eval()
for p in _clip_full.parameters():
    p.requires_grad_(False)

print("\n=== Stage 1b: Pseudo-labeling (GCN + CLIP 80-prompt zero-shot ensemble, cached) ===")
soft_labels, lp_mask, lp_hard = ensemble_pseudo_label(
    cfg, adapter, _clip_full, train_cache, ly_train, N_l_global,
    len(all_train_imgs), refresh_count=0)


## Stage 2: CLIP特徴空間での Manifold Mixup

In [ ]:
# ================================================================
# Stage 2: CLIP特徴空間での Manifold Mixup
# ----------------------------------------------------------------
# v2 では cVAE+GAN で「画像そのもの」を生成して疑似サンプルを
# 増やしていたが、CIFAR-10 の自然画像生成は難易度が高く、
# 軽量な Generator では画質が崩れ、疑似ラベル付きデータに
# ノイズを混ぜるだけになるリスクが高い。
#
# 代わりに、CLIP+Adapter の特徴空間 (画像そのものではなく
# 512次元の埋め込み) 上で Mixup を行う。特徴空間は意味的に
# 滑らかなため、ピクセル空間の補間より「ラベル間を補間した
# 合成サンプル」が意味を持ちやすい (Manifold Mixupの発想)。
# 学習コストは前向き計算1回分のみで、GAN学習のような不安定さがない。
# ============================================================

def feature_mixup(feat_a, label_a, feat_b, label_b, alpha=0.4):
    """
    feat_a, feat_b: (B, D) 特徴ベクトル (ラベル付き or 高信頼疑似ラベル)
    label_a, label_b: (B,) クラスindex (one-hotではなくhard label)
    Returns: 混合特徴, soft target (B, n_classes), lam
    """
    B = feat_a.size(0)
    lam = np.random.beta(alpha, alpha, size=B).astype(np.float32)
    lam_t = torch.tensor(lam, device=feat_a.device).unsqueeze(1)
    mixed_feat = lam_t * feat_a + (1 - lam_t) * feat_b
    return mixed_feat, label_a, label_b, lam_t.squeeze(1)


def build_mixup_batch(cfg, feats, hard_labels, confident_mask, n_classes, n_pairs):
    """
    confident なサンプル (labeled + 高信頼疑似ラベル) からランダムに2つ選び、
    feature_mixup でペアを作る。異なるクラス間のペアを優先することで、
    決定境界付近の合成サンプルを増やす (Manifold Mixupの主眼)。
    """
    conf_idx = np.where(confident_mask)[0]
    idx_a = np.random.choice(conf_idx, n_pairs)
    idx_b = np.random.choice(conf_idx, n_pairs)
    fa = feats[idx_a]; fb = feats[idx_b]
    ya = torch.as_tensor(hard_labels[idx_a], dtype=torch.long)
    yb = torch.as_tensor(hard_labels[idx_b], dtype=torch.long)
    return feature_mixup(fa, ya, fb, yb, alpha=cfg.mixup_alpha)


## Stage 3: 分類器ヘッド学習

In [ ]:
# ================================================================
# Stage 3: 分類器ヘッド学習 (v4.1: 特徴キャッシュ利用、CLIP forwardゼロ)
# ----------------------------------------------------------------
# v3/v4までは、1バッチにつき CLIP forward を 4回 (labeled CE用, 疑似ラベルCE用,
# FixMatch weak/strong用) 呼んでいたため、200epoch全体で数万回のCLIP forwardが
# 発生し、Stage 3だけで8時間程度かかっていた。
#
# v4.1: Stage 1完了後にCLIPは完全frozenであることが確定しているため、
# train_cache (identity + weak×N + strong×M の生CLIP特徴, adapt前) から
# 都度ランダムに1つを選んで使う。CLIP forwardは一切発生せず、学習ループは
# 「キャッシュからのlookup -> adapter.adapt() (軽量MLP) -> head (軽量MLP)」のみになる。
# Adapter自体は小さいMLPなので、GPU上でも無視できるコストで前向き計算できる。
# ================================================================

class ClassifierHead(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.clip_feat_dim, cfg.cls_hidden), nn.ReLU(True),
            nn.Dropout(cfg.cls_dropout),
            nn.Linear(cfg.cls_hidden, cfg.cls_hidden // 2), nn.ReLU(True),
            nn.Dropout(cfg.cls_dropout),
            nn.Linear(cfg.cls_hidden // 2, cfg.n_classes))

    def forward(self, feat):
        return self.net(feat)


def update_ema(src, tgt, decay):
    with torch.no_grad():
        for ps, pt in zip(src.parameters(), tgt.parameters()):
            pt.data.mul_(decay).add_(ps.data, alpha=1 - decay)


@torch.no_grad()
def eval_head_cached(adapter, head, cache, idx, ly, cfg):
    """cache.identity[idx] (CLIP特徴, adapt前) -> adapter.adapt() -> head で評価。"""
    head.eval()
    ce = nn.CrossEntropyLoss()
    feat = adapter.adapt(cache.identity[idx].to(cfg.device))
    logits = head(feat)
    head.train()
    return ce(logits, ly.to(cfg.device)).item(), (logits.argmax(1).cpu() == ly).float().mean().item()


def make_pseudo_dataset_idx(N_l, lp_mask, lp_hard, ly_train):
    """画像そのものではなく cache 上のインデックスと対応するラベルを返す。"""
    ul_mask = torch.tensor(lp_mask[N_l:])
    ul_idx  = torch.where(ul_mask)[0]
    idx_all = torch.cat([torch.arange(N_l), ul_idx + N_l])
    y_all   = torch.cat([ly_train, torch.tensor(lp_hard[N_l:][ul_idx.numpy()]).long()])
    return idx_all, y_all


def train_classifier(cfg, adapter, _clip_full, train_cache, val_cache,
                     ly_train, ly_val, N_l, N_total,
                     soft_labels, lp_mask, lp_hard):
    head     = ClassifierHead(cfg).to(cfg.device)
    head_ema = copy.deepcopy(head)

    # Adapterは Stage 1 で学習済み。Stage 3 では追加更新しない (frozen) ことで
    # キャッシュ済み特徴との整合性を保つ。Headのみ学習する。
    adapter.eval()
    for p in adapter.parameters():
        p.requires_grad_(False)

    opt   = torch.optim.AdamW(head.parameters(), lr=cfg.cls_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.cls_epochs)
    ce_fn = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    head_swa = AveragedModel(head)
    swa_start_ep = int(cfg.cls_epochs * cfg.swa_start)

    refresh_count = 1
    history = []
    best_val_acc = 0.0
    os.makedirs("models/sgan", exist_ok=True)

    idx_ps, y_ps_all = make_pseudo_dataset_idx(N_l, lp_mask, lp_hard, ly_train)
    n_unlabeled = N_total - N_l
    unl_idx_all = torch.arange(N_l, N_total)

    def adapted_pool():
        """mixup用: identity特徴をadapterに通した全件 (NxD)。Adapterはfrozenなのでno_grad。"""
        with torch.no_grad():
            return adapter.adapt(train_cache.identity.to(cfg.device)).cpu()

    mixup_feats = adapted_pool()

    print("\n=== Stage 3: Classifier head training (cache-based, no CLIP forward) ===")
    epoch_bar = tqdm(range(cfg.cls_epochs), desc=f"Epoch", position=0, leave=True, unit="epoch")
    for epoch in epoch_bar:
        head.train()

        if epoch > 0 and epoch % cfg.pseudo_refresh == 0:
            tqdm.write(f"  [ep {epoch}] Pseudo-label refresh #{refresh_count} ...")
            soft_labels, lp_mask, lp_hard = ensemble_pseudo_label(
                cfg, adapter, _clip_full, train_cache, ly_train, N_l, N_total,
                refresh_count=refresh_count)
            refresh_count += 1
            idx_ps, y_ps_all = make_pseudo_dataset_idx(N_l, lp_mask, lp_hard, ly_train)
            mixup_feats = adapted_pool()

        # シャッフルしたインデックスでミニバッチを回す (画像ではなくキャッシュindex単位)
        perm_order = torch.randperm(len(idx_ps))
        idx_ps_shuf = idx_ps[perm_order]
        y_ps_shuf   = y_ps_all[perm_order]

        unl_perm = unl_idx_all[torch.randperm(n_unlabeled)]
        lab_perm_pool = torch.randperm(N_l)  # labeledも毎epochシャッフル

        losses = []
        n_batches = max(1, len(idx_ps_shuf) // cfg.cls_batch)

        for bi in range(n_batches):
            s, e = bi * cfg.cls_batch, (bi + 1) * cfg.cls_batch
            b_idx_ps = idx_ps_shuf[s:e]; b_y_ps = y_ps_shuf[s:e]
            if len(b_idx_ps) == 0: continue
            B = len(b_idx_ps)

            # labeled バッチ (curriculum CE用)。labeledが尽きたら巡回。
            lab_s = (bi * cfg.cls_batch) % N_l
            b_idx_l = lab_perm_pool[lab_s: lab_s + cfg.cls_batch]
            if len(b_idx_l) < cfg.cls_batch:
                b_idx_l = torch.cat([b_idx_l, lab_perm_pool[:cfg.cls_batch - len(b_idx_l)]])
            b_y_l = ly_train[b_idx_l]

            # unlabeled バッチ (FixMatch consistency用)
            unl_s = (bi * cfg.cls_batch) % n_unlabeled
            b_idx_u = unl_perm[unl_s: unl_s + cfg.cls_batch]
            if len(b_idx_u) < cfg.cls_batch:
                b_idx_u = torch.cat([b_idx_u, unl_perm[:cfg.cls_batch - len(b_idx_u)]])

            # ----- 疑似ラベル付きデータの CE (strong variant をキャッシュから sample) -----
            feat_raw_ps = train_cache.sample_strong(b_idx_ps).to(cfg.device)
            feat_ps = adapter.adapt(feat_raw_ps)
            logits_ps = head(feat_ps)
            loss_ps = ce_fn(logits_ps, b_y_ps.to(cfg.device))

            # ----- FixMatch consistency (weakで決定 → strongで学習) -----
            with torch.no_grad():
                feat_u_weak = adapter.adapt(train_cache.sample_weak(b_idx_u).to(cfg.device))
                logits_u_weak = head(feat_u_weak)
                pseudo_prob = F.softmax(logits_u_weak, dim=-1)
            conf_mask = pseudo_prob.max(1).values >= cfg.fixmatch_thresh
            soft_sharp = F.softmax(logits_u_weak / cfg.fixmatch_temp, dim=-1)

            feat_u_strong = adapter.adapt(train_cache.sample_strong(b_idx_u).to(cfg.device))
            logits_u_strong = head(feat_u_strong)
            if conf_mask.any():
                loss_fm = F.kl_div(F.log_softmax(logits_u_strong[conf_mask], dim=-1),
                                   soft_sharp[conf_mask].clamp(1e-6), reduction="batchmean")
            else:
                loss_fm = torch.tensor(0., device=cfg.device)

            # ----- Manifold Mixup (CLIP特徴空間, キャッシュ済みadapt後特徴を使用) -----
            loss_mix = torch.tensor(0., device=cfg.device)
            if cfg.use_feature_mixup:
                n_pairs = B * cfg.mixup_pairs_per_epoch
                fa, ya, yb, lam_t = build_mixup_batch(
                    cfg, mixup_feats, lp_hard, lp_mask, cfg.n_classes, n_pairs)
                fa, ya, yb, lam_t = (fa.to(cfg.device), ya.to(cfg.device),
                                    yb.to(cfg.device), lam_t.to(cfg.device))
                logits_mix = head(fa)
                loss_mix = (lam_t * F.cross_entropy(logits_mix, ya, reduction="none") +
                           (1 - lam_t) * F.cross_entropy(logits_mix, yb, reduction="none")).mean()

            # ----- 通常labeledデータの curriculum CE (weak variantを使用) -----
            feat_raw_l = train_cache.sample_weak(b_idx_l).to(cfg.device)
            feat_cls = adapter.adapt(feat_raw_l)
            logits_cls = head(feat_cls)
            frac = epoch / max(1, cfg.cls_epochs)
            if frac >= 0.3 and random.random() < 0.5:
                lam2 = float(np.random.beta(cfg.mixup_alpha, cfg.mixup_alpha))
                perm_l = torch.randperm(B, device=cfg.device)
                logits_mixed_l = head(lam2 * feat_cls + (1 - lam2) * feat_cls[perm_l])
                loss_cls = mix_criterion(ce_fn, logits_mixed_l, b_y_l.to(cfg.device),
                                         b_y_l.to(cfg.device)[perm_l.cpu()], lam2)
            else:
                loss_cls = ce_fn(logits_cls, b_y_l.to(cfg.device))

            loss = (loss_cls + cfg.lam_pseudo * loss_ps + loss_fm +
                   cfg.lam_mixup * loss_mix)

            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), 1.)
            opt.step()
            update_ema(head, head_ema, cfg.ema_decay)
            losses.append(loss.item())
            epoch_bar.set_postfix(loss=f"{loss.item():.4f}")

        sched.step()
        if epoch >= swa_start_ep:
            head_swa.update_parameters(head)

        if (epoch + 1) % cfg.log_interval == 0:
            tr_l = np.mean(losses)
            val_l, val_acc = eval_head_cached(adapter, head, val_cache,
                                              torch.arange(len(ly_val)), ly_val, cfg)
            _, train_acc = eval_head_cached(adapter, head, train_cache,
                                            torch.arange(N_l), ly_train, cfg)
            tqdm.write(
                f"[Cls ep {epoch+1:03d}/{cfg.cls_epochs}] "
                f"train_loss={tr_l:.4f}  val_loss={val_l:.4f} | "
                f"train_acc={train_acc*100:.1f}%  val_acc={val_acc*100:.1f}%")
            history.append((epoch+1, tr_l, val_l, train_acc, val_acc))
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(head.state_dict(), "models/sgan/head_best.pth")
                torch.save(head_ema.state_dict(), "models/sgan/head_ema_best.pth")
                tqdm.write(f"  => [Best] val_acc={best_val_acc*100:.2f}%  saved.")

    torch.save(head.state_dict(), "models/sgan/head_final.pth")
    tqdm.write("\nSaving SWA stats ...")
    torch.save(head_swa.state_dict(), "models/sgan/head_swa_final.pth")
    return head, head_ema, head_swa, history


head, head_ema, head_swa, history = train_classifier(
    cfg, adapter, _clip_full, train_cache, val_cache,
    ly_train, ly_val, N_l_global, len(all_train_imgs),
    soft_labels, lp_mask, lp_hard)


## Inference: TTA + 最終評価

In [ ]:
# ================================================================
# Inference: TTA + 最終評価 (v4.1: test_cache 利用、CLIP forwardゼロ)
# ================================================================

def tta_predict_cached(adapter, head, cache, idx, device=DEVICE):
    """
    cache (test_cache) の identity + 全weak variants + 全strong variants を
    すべて使ってTTAを行う (キャッシュ済みなのでCLIP forwardは発生しない)。
    """
    head.eval()
    views = [cache.identity[idx]]
    for v in range(cache.weak.size(0)):
        views.append(cache.weak[v, idx])
    for v in range(cache.strong.size(0)):
        views.append(cache.strong[v, idx])

    probs = []
    with torch.no_grad():
        for feat_raw in views:
            feat = adapter.adapt(feat_raw.to(device))
            probs.append(F.softmax(head(feat), dim=-1))
    return torch.stack(probs).mean(0)


def evaluate_test_cached(adapter, head, cache, test_lbls, batch=256, label="", device=DEVICE):
    preds = []
    N = cache.identity.size(0)
    for i in tqdm(range(0, N, batch), desc=f"TTA [{label}]", position=0, leave=True):
        idx = torch.arange(i, min(i + batch, N))
        preds.append(tta_predict_cached(adapter, head, cache, idx, device=device).argmax(1).cpu())
    preds = torch.cat(preds)
    acc = (preds == test_lbls).float().mean().item()
    print(f"[{label}] ★ TEST Accuracy (final): {acc*100:.2f}%")
    return acc


print("\n=== Final Test Evaluation (1回のみ) ===")
if history:
    best_ep = max(history, key=lambda h: h[4])
    print(f"  Best val_acc: {best_ep[4]*100:.1f}%  @ epoch {best_ep[0]}")

evaluate_test_cached(adapter, head,     test_cache, test_lbls, label="Head (final)")
evaluate_test_cached(adapter, head_ema, test_cache, test_lbls, label="Head (EMA)")

# 学習曲線プロット
try:
    import matplotlib.pyplot as plt
    if history:
        eps        = [h[0] for h in history]
        train_ls   = [h[1] for h in history]
        val_ls     = [h[2] for h in history]
        train_accs = [h[3]*100 for h in history]
        val_accs   = [h[4]*100 for h in history]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(eps, train_ls, label="train loss", marker="o", ms=3)
        axes[0].plot(eps, val_ls,   label="val loss",   marker="s", ms=3)
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
        axes[0].set_title("Stage 3 — Loss (train vs val)")
        axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(eps, train_accs, label="train acc", marker="o", ms=3)
        axes[1].plot(eps, val_accs,   label="val acc",   marker="s", ms=3)
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
        axes[1].set_title("Stage 3 — Accuracy (train vs val)")
        axes[1].legend(); axes[1].grid(alpha=0.3)

        plt.tight_layout(); plt.show()
except ImportError:
    pass


## 比較用 Baseline (簡易, 厳密なablationではない)

In [ ]:
# ================================================================
# 比較用 Baseline (v4: SimpleCNN -> pretrained ResNet-50 fine-tune)
# ----------------------------------------------------------------
#   (1) baseline_resnet50   : ImageNet pretrained ResNet-50 を labeled 500枚で
#                              fine-tune。研究上の公平な比較対象として、
#                              事前学習あり・ラベルのみ・半教師なし手法なし、
#                              という条件を揃える。
#                              (v3のSimpleCNNより強いbaselineで、提案手法の
#                               優位性をより説得力ある形で示す)
#   (2) clip_supervised_only: CLIP+Adapter特徴 + labeled 500枚のみ
#                              (疑似ラベル・mixupなし。CLIPの素の効果のみ測る)
# ================================================================

def train_baseline_resnet50(cfg, lx_train, ly_train, lx_val, ly_val,
                            test_imgs, test_lbls, epochs=100, device=DEVICE):
    """
    pretrained ResNet-50 を labeled 500枚でfine-tune。
    入力は 224x224 (CLIPと同じ解像度) をそのまま使う。
    最終FC層のみを学習させる linear probe と、全層を低LRで学習する
    full fine-tune の2段階で行う (先にlinear probe -> 次にfull ft)。
    """
    import torchvision.models as tvm

    # ImageNet 正規化 (ResNet-50は ImageNet mean/std を期待する)
    imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
    imagenet_std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

    def normalize_imagenet(x):
        return (x.float() - imagenet_mean) / imagenet_std

    # 画像テンソルはすでに 224x224 (CLIP解像度と同じ)。そのまま使う。
    lx_t_norm = normalize_imagenet(lx_train)
    lx_v_norm = normalize_imagenet(lx_val)

    model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, cfg.n_classes)
    model = model.to(device)

    ce = nn.CrossEntropyLoss()
    loader_t = DataLoader(TensorDataset(lx_t_norm, ly_train), batch_size=32,
                          shuffle=True, num_workers=0)

    # --- Phase 1: Linear probe (FCのみ, 30 epoch) ---
    for name, p in model.named_parameters():
        p.requires_grad_("fc" in name)
    opt1 = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    for epoch in range(30):
        model.train()
        for x, y in loader_t:
            x, y = x.to(device), y.to(device)
            opt1.zero_grad(); ce(model(x), y).backward(); opt1.step()

    # --- Phase 2: Full fine-tune (全層, 低LR, 残り epochs) ---
    for p in model.parameters():
        p.requires_grad_(True)
    opt2 = torch.optim.AdamW([
        {"params": [p for n,p in model.named_parameters() if "fc" not in n], "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 1e-3},
    ], weight_decay=1e-4)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, epochs - 30)
    best_acc = 0.0
    for epoch in range(epochs - 30):
        model.train()
        for x, y in loader_t:
            x, y = x.to(device), y.to(device)
            opt2.zero_grad(); ce(model(x), y).backward(); opt2.step()
        sched2.step()
        if (epoch + 1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                pred = model(lx_v_norm.to(device)).argmax(1).cpu()
            acc = (pred == ly_val).float().mean().item()
            best_acc = max(best_acc, acc)
            tqdm.write(f"[ResNet-50 FT ep {epoch+31}] val_acc={acc*100:.1f}%")

    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(test_imgs), 128):
            x = normalize_imagenet(test_imgs[i:i+128]).to(device)
            preds.append(model(x).argmax(1).cpu())
    preds = torch.cat(preds)
    test_acc = (preds == test_lbls).float().mean().item()
    print(f"[Baseline ResNet-50 FT] ★ TEST Accuracy: {test_acc*100:.2f}%")
    return test_acc


def train_clip_supervised_only(cfg, adapter, train_cache, val_cache, test_cache,
                               ly_train, ly_val, test_lbls, N_l, epochs=100):
    """CLIP+Adapter特徴のみ、labeled 500枚のみで学習 (疑似ラベル・mixupなし)。
    キャッシュ済み特徴を使うため CLIP forward は発生しない。"""
    head = ClassifierHead(cfg).to(cfg.device)
    opt  = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
    ce   = nn.CrossEntropyLoss()

    with torch.no_grad():
        feat_train = adapter.adapt(train_cache.identity[:N_l].to(cfg.device)).cpu()
        feat_val   = adapter.adapt(val_cache.identity.to(cfg.device)).cpu()

    loader = DataLoader(TensorDataset(feat_train, ly_train), batch_size=64, shuffle=True)
    for epoch in range(epochs):
        head.train()
        for f, y in loader:
            f, y = f.to(cfg.device), y.to(cfg.device)
            opt.zero_grad(); ce(head(f), y).backward(); opt.step()
        if (epoch+1) % 20 == 0:
            head.eval()
            with torch.no_grad():
                pred = head(feat_val.to(cfg.device)).argmax(1).cpu()
            acc = (pred == ly_val).float().mean().item()
            tqdm.write(f"[CLIP supervised-only ep {epoch+1}] val_acc={acc*100:.1f}%")

    head.eval()
    with torch.no_grad():
        feat_test = adapter.adapt(test_cache.identity.to(cfg.device))
        preds = head(feat_test).argmax(1).cpu()
    test_acc = (preds == test_lbls).float().mean().item()
    print(f"[CLIP supervised-only] ★ TEST Accuracy: {test_acc*100:.2f}%")
    return test_acc


print("\n=== Reference comparisons ===")
acc_resnet50  = train_baseline_resnet50(cfg, lx_train, ly_train, lx_val, ly_val,
                                        test_imgs, test_lbls)
acc_clip_only = train_clip_supervised_only(cfg, adapter, train_cache, val_cache, test_cache,
                                           ly_train, ly_val, test_lbls, N_l_global)
acc_proposed  = evaluate_test_cached(adapter, head_ema, test_cache, test_lbls,
                                     label="Proposed (EMA)")

print("\n=== Summary ===")
print(f"  Baseline: pretrained ResNet-50 fine-tune (500 labels)  : {acc_resnet50*100:.2f}%")
print(f"  CLIP ViT-L/14 + Adapter, supervised-only (500 labels)  : {acc_clip_only*100:.2f}%")
print(f"  Proposed (GCN+ZS pseudo-label + feature mixup)         : {acc_proposed*100:.2f}%")
